<a href="https://colab.research.google.com/github/sofiiamyshelova-prog/python_for_ds_tasks/blob/main/Copy_of_HW_2_2_%D0%9D%D0%B5%D0%B7%D0%B1%D0%B0%D0%BB%D0%B0%D0%BD%D1%81%D0%BE%D0%B2%D0%B0%D0%BD%D0%B0_%D0%B1%D0%B0%D0%B3%D0%B0%D1%82%D0%BE%D0%BA%D0%BB%D0%B0%D1%81%D0%BE%D0%B2%D0%B0_%D0%BA%D0%BB%D0%B0%D1%81%D0%B8%D1%84%D1%96%D0%BA%D0%B0%D1%86%D1%96%D1%8F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

У цьому ДЗ ми потренуємось розв'язувати задачу багатокласової класифікації за допомогою логістичної регресії з використанням стратегій One-vs-Rest та One-vs-One, оцінити якість моделей та порівняти стратегії.

### Опис задачі і даних

**Контекст**

В цьому ДЗ ми працюємо з даними про сегментацію клієнтів.

Сегментація клієнтів – це практика поділу бази клієнтів на групи індивідів, які схожі між собою за певними критеріями, що мають значення для маркетингу, такими як вік, стать, інтереси та звички у витратах.

Компанії, які використовують сегментацію клієнтів, виходять з того, що кожен клієнт є унікальним і що їхні маркетингові зусилля будуть більш ефективними, якщо вони орієнтуватимуться на конкретні, менші групи зі зверненнями, які ці споживачі вважатимуть доречними та які спонукатимуть їх до купівлі. Компанії також сподіваються отримати глибше розуміння уподобань та потреб своїх клієнтів з метою виявлення того, що кожен сегмент цінує найбільше, щоб точніше адаптувати маркетингові матеріали до цього сегменту.

**Зміст**.

Автомобільна компанія планує вийти на нові ринки зі своїми існуючими продуктами (P1, P2, P3, P4 і P5). Після інтенсивного маркетингового дослідження вони дійшли висновку, що поведінка нового ринку схожа на їхній існуючий ринок.

На своєму існуючому ринку команда з продажу класифікувала всіх клієнтів на 4 сегменти (A, B, C, D). Потім вони здійснювали сегментовані звернення та комунікацію з різними сегментами клієнтів. Ця стратегія працювала для них надзвичайно добре. Вони планують використати ту саму стратегію на нових ринках і визначили 2627 нових потенційних клієнтів.

Ви маєте допомогти менеджеру передбачити правильну групу для нових клієнтів.

В цьому ДЗ використовуємо дані `customer_segmentation_train.csv`[скачати дані](https://drive.google.com/file/d/1VU1y2EwaHkVfr5RZ1U4MPWjeflAusK3w/view?usp=sharing). Це `train.csv`з цього [змагання](https://www.kaggle.com/datasets/abisheksudarshan/customer-segmentation/data?select=train.csv)

**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [35]:
import pandas as pd
customer_segmentation_train = pd.read_csv('/content/drive/MyDrive/ML/data/customer_segmentation_train.csv', index_col=0)

customer_segmentation_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8068 entries, 462809 to 461879
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Gender           8068 non-null   object 
 1   Ever_Married     7928 non-null   object 
 2   Age              8068 non-null   int64  
 3   Graduated        7990 non-null   object 
 4   Profession       7944 non-null   object 
 5   Work_Experience  7239 non-null   float64
 6   Spending_Score   8068 non-null   object 
 7   Family_Size      7733 non-null   float64
 8   Var_1            7992 non-null   object 
 9   Segmentation     8068 non-null   object 
dtypes: float64(2), int64(1), object(7)
memory usage: 693.3+ KB


In [39]:
df = customer_segmentation_train.copy()

df['isFemale'] = df['Gender'].map({'Male': 0, 'Female': 1})
df['isEver_Married'] = df['Ever_Married'].map({'No': 0, 'Yes': 1}).fillna(0).astype(int)
df['isGraduated'] = df['Graduated'].map({'No': 0, 'Yes': 1}).fillna(0).astype(int)

df['Spending_Low'] = (df['Spending_Score'] == 'Low').astype(int)
df['Spending_Average'] = (df['Spending_Score'] == 'Average').astype(int)
df['Spending_High'] = (df['Spending_Score'] == 'High').astype(int)

df['Work_Experience'] = df['Work_Experience'].fillna(0.0) #порожній досвід замінюємо на 0
df['Family_Size'] = df['Family_Size'].fillna(1.0) #порожні поля замінюємо на 1
df['Var_1_int'] = df['Var_1'].map({'Cat_1':1, 'Cat_2':2,'Cat_3':3,'Cat_4':4,'Cat_5':5,'Cat_6':6,}).fillna(6.0).astype(int) #порожні заміняємо на 6, бо його найбільше
df['Profession'] = df['Profession'].fillna('Unknown')


df = df.drop(columns=['Gender', 'Ever_Married', 'Graduated', 'Spending_Score', 'Var_1'])


df.head()


,Age,Profession,Work_Experience,Family_Size,Segmentation,isFemale,isEver_Married,isGraduated,Spending_Low,Spending_Average,Spending_High,Var_1_int
ID,,,,,,,,,,,,
462809,22,Healthcare,1.0,4.0,D,0,0,0,1,0,0,4
462643,38,Engineer,0.0,3.0,A,1,1,1,0,1,0,4
466315,67,Engineer,1.0,1.0,B,1,1,1,1,0,0,6
461735,67,Lawyer,0.0,2.0,B,0,1,1,0,0,1,6
462669,40,Entertainment,0.0,6.0,A,1,1,1,0,0,1,6


In [38]:
from sklearn.model_selection import train_test_split

X=df.drop(columns=['Segmentation'])
y=df['Segmentation']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)


**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

In [49]:
cat_feature_indices = [X_train.columns.get_loc('Profession')]

from imblearn.over_sampling import SMOTENC

smote_nc = SMOTENC(
    categorical_features=cat_feature_indices,
    random_state=42)

X_smote, y_smote = smote_nc.fit_resample(X_train, y_train)

print('До SMOTENC:')
print(X_train.shape)
print(y_train.value_counts())

print('Після SMOTENC:')
print(X_smote.shape)
print(y_smote.value_counts())

До SMOTENC:
(6454, 11)
Segmentation
D    1814
A    1578
C    1576
B    1486
Name: count, dtype: int64
Після SMOTENC:
(7256, 11)
Segmentation
A    1814
B    1814
C    1814
D    1814
Name: count, dtype: int64


In [50]:
from sklearn.preprocessing import LabelEncoder
le_prof = LabelEncoder()
X_train['Profession'] = le_prof.fit_transform(X_train['Profession'])


from imblearn.combine import SMOTETomek

smote_tomek = SMOTETomek(
    smote=SMOTENC(
        categorical_features=cat_feature_indices,
        random_state=42),
    random_state=42)

X_smote_tomek, y_smote_tomek = smote_tomek.fit_resample(X_train, y_train)

print('Після SMOTE-Tomek:')
print(X_smote_tomek.shape)
print(y_smote_tomek.value_counts())

Після SMOTE-Tomek:
(6202, 11)
Segmentation
C    1581
D    1577
B    1545
A    1499
Name: count, dtype: int64


**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

In [53]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_cols = ['Age',
                'Work_Experience',
                'Family_Size',
                'Var_1_int',
                'isFemale',
                'isEver_Married',
                'isGraduated',
                'Spending_Low',
                'Spending_Average',
                'Spending_High']
categorical_cols = ['Profession']

def model():
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
        ]
    )

    return Pipeline([
        ('preprocessor', preprocessor),
        ('clf', OneVsRestClassifier(
            LogisticRegression(max_iter=2000, random_state=42)
        ))
    ])


# ---------- 1. Оригінальні дані ----------
model_original = model()
model_original.fit(X_train, y_train)
pred_original = model_original.predict(X_test)
print('ORIGINAL DATA')
print(classification_report(y_test, pred_original))

# ---------- 2. SMOTE ----------
model_smote = model()
model_smote.fit(X_smote, y_smote)
pred_smote = model_smote.predict(X_test)
print('SMOTE')
print(classification_report(y_test, pred_smote))

# ---------- 3. SMOTE-Tomek ----------
model_smote_tomek = model()
model_smote_tomek.fit(X_smote_tomek, y_smote_tomek)
pred_smote_tomek = model_smote_tomek.predict(X_test)
print('SMOTE-TOMEK')
print(classification_report(y_test, pred_smote_tomek))

ORIGINAL DATA
              precision    recall  f1-score   support

           A       0.37      0.22      0.28       394
           B       0.32      0.09      0.14       372
           C       0.48      0.61      0.54       394
           D       0.49      0.84      0.62       454

    accuracy                           0.46      1614
   macro avg       0.42      0.44      0.39      1614
weighted avg       0.42      0.46      0.40      1614

SMOTE
              precision    recall  f1-score   support

           A       0.41      0.47      0.44       394
           B       0.39      0.24      0.29       372
           C       0.52      0.61      0.56       394
           D       0.67      0.69      0.68       454

    accuracy                           0.51      1614
   macro avg       0.50      0.50      0.49      1614
weighted avg       0.50      0.51      0.50      1614

SMOTE-TOMEK
              precision    recall  f1-score   support

           A       0.37      0.22      0.28

Для порівняння моделей було обрано F1-score, оскільки задача є багатокласовою та містить незбалансовані класи.

Найкращі результати показала модель логістичної регресії з використанням SMOTE. Вона досягла accuracy 0.51 та macro F1-score 0.49, що є вищими за результати моделі на оригінальних даних (macro F1 = 0.39) та моделі після SMOTE-Tomek (macro F1 = 0.40). Для класів A та B, які були менш представлені у вибірці, є відчутне покращення.

Модель зі SMOTE-Tomek не дала суттєвого покращення порівняно з оригінальною моделлю. Можливо, метод видалив частину корисних синтетичних прикладів, створених SMOTE, через сильне перекриття меж між класами. Тому для цього набору даних простий SMOTE виявився більш ефективним, ніж комбінований підхід SMOTE-Tomek.